<a href="https://colab.research.google.com/github/vietnamdripvd-oss/shorts3d/blob/main/render_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Render EEVEE (toon) di Colab GPU

1. **Runtime → Change runtime type → T4 GPU**.
2. Repo private → simpan GitHub token (fine-grained, *Contents: Read-only* untuk repo ini) di
   **Colab Secrets** (ikon kunci di sidebar kiri) dengan nama `GH_TOKEN`, nyalakan *Notebook access*.
3. Atur cell **Config**, lalu **Runtime → Run all**.
4. Sesi putus? Sambung lagi, *Run all* lagi — frame yang sudah ada di Google Drive di-skip.

In [1]:
#@title Config
REPO       = "vietnamdripvd-oss/shorts3d"  #@param {type:"string"}
BRANCH     = "main"   #@param {type:"string"}
SLUG       = "test_e2e"            #@param {type:"string"}
BUILD_SCRIPT = "test_e2e/scene.py" #@param {type:"string"}
RES_PCT    = 50    #@param {type:"integer"}  50 = 540x960 (tes/preview), 100 = 1080x1920 (final)
SAMPLES    = 8     #@param {type:"integer"}  8 tes, 16 final
CHUNK      = 30    #@param {type:"integer"}
AUDIO      = ""    #@param {type:"string"}   wav relatif repo, kosong = bisu
USE_DRIVE  = True  #@param {type:"boolean"}
BLENDER_VER = "4.2.3"

In [2]:
#@title 1. GPU + Google Drive
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/shorts3d_render'
else:
    WORK = '/content/shorts3d_render'
os.makedirs(WORK, exist_ok=True)
print('WORK =', WORK)

name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07
Mounted at /content/drive
WORK = /content/drive/MyDrive/shorts3d_render


In [3]:
#@title 2. Install dependency + EGL NVIDIA (biar EEVEE pakai GPU)
!apt-get -qq update > /dev/null
!apt-get -qq install -y libegl1 libgl1 libglvnd0 libxi6 libxkbcommon0 libxrender1 libsm6 libxxf86vm1 libxfixes3 ffmpeg > /dev/null
import os, glob, json
print('libEGL_nvidia:', glob.glob('/usr/lib64-nvidia/libEGL_nvidia.so*') + glob.glob('/usr/lib/x86_64-linux-gnu/libEGL_nvidia.so*'))
os.makedirs('/usr/share/glvnd/egl_vendor.d', exist_ok=True)
VENDOR = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
with open(VENDOR, 'w') as f:
    json.dump({"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}, f)
os.environ['LD_LIBRARY_PATH'] = '/usr/lib64-nvidia:' + os.environ.get('LD_LIBRARY_PATH', '')
os.environ['__EGL_VENDOR_LIBRARY_FILENAMES'] = VENDOR

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
libEGL_nvidia: ['/usr/lib64-nvidia/libEGL_nvidia.so.0', '/usr/lib64-nvidia/libEGL_nvidia.so.580.82.07']


In [4]:
#@title 3. Install Blender portable (tarball di-cache di Drive)
mm = '.'.join(BLENDER_VER.split('.')[:2])
tar = f'{WORK}/blender-{BLENDER_VER}-linux-x64.tar.xz'
if not os.path.exists(tar):
    !wget -q -O "{tar}" https://download.blender.org/release/Blender{mm}/blender-{BLENDER_VER}-linux-x64.tar.xz
if not os.path.exists(f'/content/blender-{BLENDER_VER}-linux-x64'):
    !tar xf "{tar}" -C /content
BLENDER = f'/content/blender-{BLENDER_VER}-linux-x64/blender'
!{BLENDER} --version | head -1

Blender 4.2.3 LTS


In [5]:
#@title 4. Clone repo (sparse: cuma folder kode, render lama nggak ikut di-download)
try:
    from google.colab import userdata
    TOKEN = userdata.get('GH_TOKEN')
except Exception:
    TOKEN = ''
url = f'https://{TOKEN}@github.com/{REPO}.git' if TOKEN else f'https://github.com/{REPO}.git'
R = '/content/repo'
if os.path.exists(f'{R}/.git'):
    !cd {R} && git fetch -q --depth 1 origin {BRANCH} && git reset -q --hard FETCH_HEAD
else:
    !git clone -q --depth 1 --filter=blob:none --sparse -b {BRANCH} "{url}" {R}
!cd {R} && git sparse-checkout set lib3d tools models rigs {SLUG} && git log --oneline -1

remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 3 (delta 0), reused 2 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), 5.55 KiB | 5.55 MiB/s, done.
remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 18 (delta 0), reused 18 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (18/18), 14.14 KiB | 14.14 MiB/s, done.
1f9220f (grafted, HEAD -> main, origin/main, origin/HEAD) Notebook Colab low-res default + generator, arahkan ke repo shorts3d


In [6]:
#@title 5. Cek GPU dipakai EEVEE (harus NVIDIA/Tesla, BUKAN llvmpipe)
out = !{BLENDER} -b --factory-startup -P {R}/tools/gpu_info.py 2>&1 | grep -E "\[gpu\]|EGL"
print('\n'.join(out))
if not any('NVIDIA' in l or 'Tesla' in l for l in out):
    print('\n!! EEVEE TIDAK pakai GPU NVIDIA. Cek Runtime = GPU, restart runtime, Run all lagi.')

[gpu] vendor  : NVIDIA Corporation
[gpu] renderer: Tesla T4/PCIe/SSE2
[gpu] version : 4.6.0 NVIDIA 580.82.07


In [7]:
#@title 6. Build scene (.blend) dari script
BLEND = f'{WORK}/{SLUG}/scene.blend'
os.makedirs(f'{WORK}/{SLUG}', exist_ok=True)
!cd {R} && {BLENDER} -b -P {BUILD_SCRIPT} -- --out "{BLEND}" 2>&1 | grep -E "TRIS|saved|Error|Traceback"

TRIS mascot: 4416
TRIS total: 4816
saved /content/drive/MyDrive/shorts3d_render/test_e2e/scene.blend


In [8]:
#@title 7. Render per chunk (RESUMABLE) + waktu per frame
import subprocess, time, json, glob, statistics
FR = f'{WORK}/{SLUG}/frames_{RES_PCT}_s{SAMPLES}'
os.makedirs(FR, exist_ok=True)
info = subprocess.run([BLENDER, '-b', BLEND, '--python-expr',
    'import bpy;s=bpy.context.scene;print("RANGE",s.frame_start,s.frame_end)'], capture_output=True, text=True).stdout
fs, fe = [int(x) for x in [l for l in info.splitlines() if l.startswith('RANGE')][0].split()[1:]]
todo = [f for f in range(fs, fe + 1) if not os.path.exists(f'{FR}/{f:05d}.png')]
print(f'frame {fs}-{fe}: {fe-fs+1-len(todo)} sudah ada, {len(todo)} sisa')
t0 = time.time()
for a in range(fs, fe + 1, CHUNK):
    b = min(a + CHUNK - 1, fe)
    if all(os.path.exists(f'{FR}/{f:05d}.png') for f in range(a, b + 1)):
        continue
    p = subprocess.run([BLENDER, '-b', BLEND, '-P', f'{R}/tools/render_frames.py', '--',
                        '--out', FR, '--start', str(a), '--end', str(b),
                        '--pct', str(RES_PCT), '--samples', str(SAMPLES)], capture_output=True, text=True)
    if p.returncode != 0:
        print(p.stdout[-2000:], p.stderr[-2000:]); raise SystemExit('render gagal')
    done = len(glob.glob(f'{FR}/*.png'))
    print(f'  chunk {a}-{b} ok | {done}/{fe-fs+1} frame | {time.time()-t0:.0f}s')
secs = [json.loads(l)['sec'] for l in open(f'{FR}/timing.jsonl')]
if secs:
    print(f'\nWAKTU/FRAME ({RES_PCT}%, {SAMPLES} samples): median {statistics.median(secs):.2f}s, '
          f'min {min(secs):.2f}s, max {max(secs):.2f}s ({len(secs)} frame)')

frame 1-150: 0 sudah ada, 150 sisa
  chunk 1-30 ok | 30/150 frame | 40s
  chunk 31-60 ok | 60/150 frame | 53s
  chunk 61-90 ok | 90/150 frame | 64s
  chunk 91-120 ok | 120/150 frame | 75s
  chunk 121-150 ok | 150/150 frame | 87s

WAKTU/FRAME (50%, 8 samples): median 0.32s, min 0.29s, max 29.32s (150 frame)


In [9]:
#@title 8. Encode mp4 (+ audio kalau ada) & download
os.makedirs(f'{WORK}/{SLUG}/render', exist_ok=True)
MP4 = f'{WORK}/{SLUG}/render/{SLUG}_{RES_PCT}.mp4'
aud = f'{R}/{AUDIO}' if AUDIO else ''
!bash {R}/tools/encode.sh "{FR}" "{MP4}" {aud}
!ffprobe -v error -show_entries stream=width,height,r_frame_rate:format=duration -of compact "{MP4}"
from google.colab import files
files.download(MP4)

-> /content/drive/MyDrive/shorts3d_render/test_e2e/render/test_e2e_50.mp4 (5.000000s)
stream|width=540|height=960|r_frame_rate=30/1
format|duration=5.000000


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>